## **Kindle Review Sentiment Analysis**

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("all_kindle_review.csv")

In [3]:
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [4]:
## Preprocessing and cleaning
## Train Test Split
## BOW, TFIDF, Word2Vec
## Train ML algorithms

In [5]:
df = df[['reviewText', 'rating']]

In [6]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [7]:
df.shape

(12000, 2)

In [8]:
## Missing Values

df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [9]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [10]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [11]:
## Preprocessing and cleaning

df['rating'] = df['rating'].apply((lambda x:0 if x < 3 else 1))

In [12]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",1
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,1
3,Aggie is Angela Lansbury who carries pocketboo...,1
4,I did not expect this type of book to be in li...,1


In [13]:
df['rating'].unique()

array([1, 0])

In [14]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [15]:
## lower all the cases

df['reviewText'] = df['reviewText'].str.lower()

In [16]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [17]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import re
from nltk.corpus import stopwords
from bs4 import BeautifulSoup

In [19]:
## Clean the data

## Removing special chracters
df['reviewText'] = df['reviewText'].apply(lambda x:re.sub('[^a-z A-z 0-9-]+', '',x))

## Remove the stopswords
df['reviewText'] = df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))

## Remove url 
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))

## Remove html tags
df['reviewText'] = df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())

## Remove any additional spaces
df['reviewText'] = df['reviewText'].apply(lambda x: " ".join(x.split()))

FeatureNotFound: Couldn't find a tree builder with the features you requested: lxml. Do you need to install a parser library?

In [ ]:
df.head()

In [ ]:
## Lemmatizer

from nltk.stem import WordNetLemmatizer

In [ ]:
lemmatizer = WordNetLemmatizer()

In [ ]:
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])


In [ ]:
df['reviewText'] = df['reviewText'].apply(lambda x:lemmatize_words(x))

In [ ]:
df.head()

In [ ]:
## Train Test Split
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(df['reviewText'], df['rating'], test_size=0.20)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

In [ ]:
X_train_bow

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb_model_bow = GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf = GaussianNB().fit(X_train_tfidf,y_train)

In [ ]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

In [ ]:
y_pred_bow = nb_model_bow.predict(X_test_bow)

In [ ]:
y_pred_tfidf = nb_model_bow.predict(X_test_tfidf)

In [ ]:
confusion_matrix(y_test,y_pred_bow)

In [ ]:
print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))

In [ ]:
confusion_matrix(y_test,y_pred_tfidf)

In [ ]:
print("TFIDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))